# Mah et al. (2023), Figure 4: training, from the paper's Zenodo deposit

**Paper.** Mah A, Schiereck SS, Bossio V, Constantinople CM (2023). Distinct value computations
support rapid sequential decisions. *Nature Communications* 14:7573.
[doi:10.1038/s41467-023-43250-x](https://doi.org/10.1038/s41467-023-43250-x). Full text used
(open access, PMC10663503), with the paper's code (github.com/constantinoplelab/published,
`rat_behavior`, commit b7b68ca).

**Data.** Figure 4 uses every session of the final training stage (`A_Structs_all`), including
the early sessions that the inclusion criteria remove. The dandiset
([DANDI:001169](https://dandiarchive.org/dandiset/001169/0.260825.1909), version 0.260825.1909)
holds only the included sessions (`A_Structs_Final`), so these claims are tested on the structs the
paper deposited on Zenodo ([10.5281/zenodo.10031483](https://doi.org/10.5281/zenodo.10031483),
version 1), read from its zip archive with range requests and held in memory (about 0.5 GB). The
dandiset is used only to count its sessions per rat. The other claims are in
`reproduce_mah_2023.ipynb`. No random numbers are drawn here.

| Claim | Data | Status |
|---|---|---|
| fig4a-wait-ratio-learning | elsewhere (Zenodo) | tested |
| fig4bc-regression-learning | elsewhere (Zenodo) | tested |
| fig4de-overshoot-learning | dandiset | blocked (data missing; see the main notebook) |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import wagering as w

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)

started = time.time()
structs = w.load_zenodo_structs("A_Structs_all")
print(f"read {len(structs)} structs from Zenodo in {time.time() - started:.0f} s")
assets = w.asset_list()

## Data check

In [ ]:
names = sorted(structs)
info = pd.DataFrame({r: {"sessions_all": len(np.atleast_1d(structs[r]["ntrials"])),
                         "trials_all": len(structs[r]["reward"]),
                         "trials_without_block": int(np.isnan(structs[r]["block"].astype(float)).sum()),
                         "stages": "/".join(str(int(s)) for s in np.unique(structs[r]["trainingstage"][~np.isnan(structs[r]["trainingstage"].astype(float))]))}
                     for r in names}).T
info["sessions_dandiset"] = assets.path.str.split("/").str[0].str[4:].value_counts().reindex(names).fillna(0).astype(int)
print("Rats:", len(names), "; same rats as the dandiset:", sorted(names) == sorted(assets.path.str.split("/").str[0].str[4:].unique()))
print(info[["sessions_all", "sessions_dandiset", "trials_all", "trials_without_block"]].astype(float).describe().loc[["min", "50%", "max"]].to_string())
print("Training stage codes present, rats:", info.stages.value_counts().to_dict())
print("Rats with more sessions here than in the dandiset:", int((info.sessions_all > info.sessions_dandiset).sum()),
      "; fewer:", int((info.sessions_all < info.sessions_dandiset).sum()))
print("Rats with at least 50 sessions here:", int((info.sessions_all >= 50).sum()))
print("Reward volumes:", sorted(np.unique(np.concatenate([structs[r]["reward"] for r in names])).tolist()))
print("Block codes:", sorted(np.unique(np.concatenate([structs[r]["block"].astype(float) for r in names])).tolist()))

The deposit holds all 291 rats. It has more sessions than the dandiset for 269 rats, as the
Zenodo README leads one to expect: these structs keep every session that ended in the final stage,
before the inclusion criteria, and their first sessions hold trials of earlier stages. For 22 rats
it has fewer sessions than the dandiset, which that description does not explain. A few trials
(at most 5 per rat) have no block; they are left out of the block measures, as the code's
regressions drop rows with NaN. Stage 9 is the paper's stage 8 (README).

In [ ]:
started = time.time()
pairs = {r: w.training_pairs(structs[r]) for r in names}
print(f"measures for {len(pairs)} rats in {time.time() - started:.0f} s")
first = {m: np.array([pairs[r][0][m].to_numpy() for r in names]) for m in pairs[names[0]][0].columns}
last = {m: np.array([pairs[r][1][m].to_numpy() for r in names]) for m in pairs[names[0]][0].columns}

def population(x):
    # Figure4.m: per-rat smoothing over groups (span 5), outliers removed, then the mean over rats.
    sm = np.array([w.remove_outliers(w.matlab_smooth(row, 5)) for row in x])
    return np.nanmean(sm, 0)

## fig4a-wait-ratio-learning

> Over training in the final stage, the wait time ratio (20 uL, high/low) fell gradually toward
> its final value, while the trial initiation time ratio was below 1 from the first session.

In [ ]:
wt1, it1 = first["wt_ratio"][:, 0], first["iti_ratio"][:, 0]
wt_late = np.nanmean(first["wt_ratio"][:, 15:25], 1)
print(f"Initiation time ratio, sessions 1 and 2: median {np.nanmedian(it1):.3f} over {np.sum(~np.isnan(it1))} rats; below 1 in "
      f"{np.sum(it1 < 1)}; signed-rank against 1 p = {w.signrank(it1 - 1):.3g}")
d = wt1 - wt_late
print(f"Wait time ratio, sessions 1 and 2: median {np.nanmedian(wt1):.3f}; sessions 31 to 50 (mean of pairs 16 to 25): median "
      f"{np.nanmedian(wt_late):.3f}; first above later in {np.sum(d > 0)} of {np.sum(~np.isnan(d))} rats; signed-rank p = {w.signrank(d):.3g}")
print(f"Wait time ratio, sessions 1 and 2, below 1 in {np.sum(wt1 < 1)} of {np.sum(~np.isnan(wt1))} rats; signed-rank against 1 p = {w.signrank(wt1 - 1):.3g}")
print("Rats per pair with a wait time ratio:", np.sum(~np.isnan(first["wt_ratio"]), 0).tolist())
print("Medians per pair, wait:", np.round(np.nanmedian(first["wt_ratio"], 0), 3).tolist())
print("Medians per pair, initiation:", np.round(np.nanmedian(first["iti_ratio"], 0), 3).tolist())
print("Last 50 sessions, medians per pair, wait:", np.round(np.nanmedian(last["wt_ratio"], 0), 3).tolist())
pw, pi_ = population(first["wt_ratio"]), population(first["iti_ratio"])
xs = np.arange(2, 51, 2)
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(xs, pw, color="#8B6DB0", label="wait time"); ax.plot(xs, pi_, color="#5DBD77", label="trial initiation time")
ax.axhline(1, color="k", lw=0.8); ax.set_xlabel("training session (pairs)"); ax.set_ylabel("ratio high/low"); ax.legend()
plt.tight_layout(); plt.show()
print("Plotted (Figure4.m's smoothing and outlier removal): wait", np.round(np.nanmin(pw), 3), "to", np.round(np.nanmax(pw), 3),
      "; initiation", np.round(np.nanmin(pi_), 3), "to", np.round(np.nanmax(pi_), 3))

**Result: partially reproduced.** The initiation time ratio is below 1 in the first two sessions
(median 0.749, below 1 in 107 of 145 rats, p = 5.4e-07). The wait time ratio of the first two
sessions (median 0.936) is not higher than over sessions 31 to 50 (median 0.917; higher in 64 of
126 rats, p = 0.23), and does not differ from 1 (p = 0.87), so the gradual fall is not shown by the
criterion's test; the medians per pair fall slowly, to 0.86 to 0.89 in the last pairs.

## fig4bc-regression-learning

> Over training, the block and reward coefficients for wait times grew gradually, while the
> previous-reward coefficient for trial initiation times was comparable early and late.

In [ ]:
for m, label in (("b_block", "wait time, block"), ("b_reward", "wait time, reward")):
    a, b = np.abs(first[m][:, 0]), np.abs(np.nanmean(first[m][:, 15:25], 1))
    print(f"|{label}| coefficient: sessions 1 and 2 median {np.nanmedian(a):.3f}; sessions 31 to 50 median {np.nanmedian(b):.3f}; "
          f"larger later in {np.sum(b > a)} of {np.sum(~np.isnan(a - b))} rats; signed-rank p = {w.signrank(b - a):.3g}")
    print("   medians per pair:", np.round(np.nanmedian(first[m], 0), 3).tolist())
e, l = np.nanmean(first["b_iti_prev"]), np.nanmean(last["b_iti_prev"])
print(f"Initiation time, previous reward coefficient: mean over the first 50 sessions {e:.4f}, over the last 50 {l:.4f}, ratio {l / e:.3f}")
pe, pl = np.nanmean(first["b_iti_prev"], 1), np.nanmean(last["b_iti_prev"], 1)
print(f"   per rat, first against last 50 (rats with both): signed-rank p = {w.signrank(pl - pe):.3g}; medians {np.nanmedian(pe):.4f} and {np.nanmedian(pl):.4f}")
print("   medians per pair, first 50:", np.round(np.nanmedian(first["b_iti_prev"], 0), 4).tolist())

**Result: reproduced.** The wait time coefficients grow with training: the block coefficient in
magnitude from a median 0.110 in sessions 1 and 2 to 0.211 over sessions 31 to 50 (larger in 144 of
205 rats, p = 9.3e-08), the reward coefficient from 0.072 to 0.148 (148 of 205, p = 5.8e-09). The
initiation time coefficient of the previous reward has the same mean over the first and the last
50 sessions (-0.0051 and -0.0051, ratio 0.99).

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig4a-wait-ratio-learning | partially reproduced |
| fig4bc-regression-learning | reproduced |
| fig4de-overshoot-learning | blocked (data missing) |

**How the analysis differs from the paper's.** The measures follow macro_regress_overtraining_V2.m
and regress_trials_blocksv2.m; the tests compare per-rat values of the first session pair with
sessions 31 to 50, which the paper does not test. The code's smoothing of each rat's coefficients
applies only to the block coefficient (a loop variable left from an earlier loop); the tests here
use unsmoothed values.

**What was found.** The early final-stage sessions behind Figure 4a to 4c are not in the dandiset;
they are public on Zenodo, where these claims were tested. The structs there include trials of
earlier training stages, as the Zenodo README says.